In [ ]:
!pip install fasttext lightautoml "lightautoml[nlp]"

In [ ]:
pip install catboost

In [ ]:
import transformers
import lightautoml
print("transformers:", transformers.__version__)
print("lightautoml:", lightautoml.__version__)

!pip uninstall -y transformers tokenizers
!pip install -U "LightAutoML==0.4.2" "transformers==4.49.0" "tokenizers<0.22"

transformers: 4.49.0
lightautoml: 0.4.2
Found existing installation: transformers 4.49.0
Uninstalling transformers-4.49.0:
  Successfully uninstalled transformers-4.49.0
Found existing installation: tokenizers 0.21.4
Uninstalling tokenizers-0.21.4:
  Successfully uninstalled tokenizers-0.21.4


In [ ]:
import re
import string
import warnings
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from catboost import CatBoostClassifier
from catboost import CatBoostRegressor, Pool

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
from catboost import CatBoostRegressor, Pool
import holidays

In [ ]:
from lightautoml.automl.presets.text_presets import TabularNLPAutoML
from lightautoml.tasks import Task

In [ ]:
PATH = "posts_vk.csv"

df = pd.read_csv(PATH, engine= "python")
df["text"] = df["text"].fillna("").astype(str)
df = df.query("text != ''")
print(df.shape)
display(df.head())
display(df.info())

In [ ]:
result = df.groupby('domain')['views'].agg(['mean', 'max', 'min', 'count'])
result

In [ ]:
def filter_out_iqr(group: pd.DataFrame) -> pd.DataFrame:
    q1 = group["views"].quantile(0.25)
    q3 = group["views"].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return group[(group["views"] >= lower) & (group["views"] <= upper)]


def clean_text_minimal(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.strip()

    return text


def base_preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "Unnamed: 0" in df.columns:
        df = df.drop(columns="Unnamed: 0")

    df["text"] = df["text"].apply(clean_text_minimal)
    df["dt_msk"] = pd.to_datetime(df["dt_msk"], errors="coerce")

    years = df["dt_msk"].dt.year.dropna().astype(int).unique().tolist()
    if years:
        ru_holidays = holidays.Russia(years=years)
        df["is_holiday"] = df["dt_msk"].dt.date.apply(
            lambda x: int(x in ru_holidays) if pd.notna(x) else 0
        )
    else:
        df["is_holiday"] = 0

    if {"domain", "views"}.issubset(df.columns):
        df = (
            df.groupby("domain", group_keys=False)
            .apply(filter_out_iqr)
            .reset_index(drop=True)
        )

    df = df.query("text != ''").copy()

    df["target_views"] = np.log1p(df["views"])

    df["engagement_rate"] = (
        (df["likes"].fillna(0) + df["comments"].fillna(0) + df["reposts"].fillna(0))
        / df["views"].replace(0, np.nan)
        * 100
    )

    drop_cols = [
        "post_type",
        "is_donut",
        "post_url",
        "marked_as_ads",
        "post_source_type",
        "views",
        "likes",
        "comments",
        "reposts",
        "reactions_total",
        "owner_id",
        "post_key",
        "from_id",
        "date_unix",
        "edited_dt_msk",
        "age_days",
        "cover_photo_url",
        "all_photo_urls",
        "is_pinned"
    ]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df.reset_index(drop=True)
    return df

In [ ]:
df = base_preprocessing(df)

In [ ]:
df.head()

In [ ]:
df['is_holiday'].unique()

## Подготовка фичей

In [ ]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    dt = pd.to_datetime(df["dt_msk"], errors="coerce")

    df["hour"] = dt.dt.hour.fillna(0).astype("int16")
    df["weekday"] = dt.dt.weekday.fillna(0).astype("int16")
    df["month"] = dt.dt.month.fillna(0).astype("int16")
    df["day"] = dt.dt.day.fillna(0).astype("int16")

    df["is_weekend"] = (df["weekday"] >= 5).astype("int8")
    df["is_workday"] = (df["weekday"] < 5).astype("int8")
    df["is_friday"] = (df["weekday"] == 4).astype("int8")

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
    df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)

    df["n_photos"] = df["n_photos"].fillna(0).astype("int16")

    txt = df["text"].fillna("").astype(str)

    df["text_len_chars"] = txt.str.len().astype("int32")
    df["text_len_words"] = txt.str.split().str.len().fillna(0).astype("int32")
    df["num_lines"] = (txt.str.count("\n") + 1).astype("int16")

    df["num_exclam"] = txt.str.count("!").astype("int16")
    df["num_question"] = txt.str.count(r"\?").astype("int16")
    df["num_dots"] = txt.str.count(r"\.").astype("int16")
    df["num_commas"] = txt.str.count(",").astype("int16")
    df["num_colons"] = txt.str.count(":").astype("int16")

    df["num_hashtags"] = txt.str.count("#").astype("int16")
    df["num_links"] = (txt.str.count("http") + txt.str.count("vk.cc")).astype("int16")
    df["has_mention"] = txt.str.contains(r"\[club|\[id|@", regex=True, na=False).astype(
        "int8"
    )
    df["num_emojis"] = txt.str.count(r"[\U0001F300-\U0001FAFF]").astype("int16")

    letters = txt.str.findall(r"[A-Za-zА-Яа-яЁё]")
    caps = txt.str.findall(r"[A-ZА-ЯЁ]")
    df["caps_ratio"] = (
        (caps.str.len() / letters.str.len().replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )
    words = txt.str.findall(r"\w+", flags=re.UNICODE)
    digits = txt.str.findall(r"\d")
    caps_words = txt.str.findall(r"\b[A-ZА-ЯЁ]{2,}\b")

    df["unique_words_count"] = words.apply(
        lambda x: len(set(w.lower() for w in x))
    ).astype("int32")

    df["unique_words_ratio"] = (
        (df["unique_words_count"] / df["text_len_words"].replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )

    df["avg_word_len"] = words.apply(
        lambda x: np.mean([len(w) for w in x]) if len(x) else 0
    ).astype("float32")

    df["long_words_count"] = words.apply(lambda x: sum(len(w) >= 8 for w in x)).astype(
        "int16"
    )

    df["digit_count"] = digits.str.len().astype("int16")
    df["digit_ratio"] = (
        (df["digit_count"] / df["text_len_chars"].replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )

    df["uppercase_words_count"] = caps_words.str.len().astype("int16")

    sentence_count = txt.str.count(r"[.!?]+")
    df["sentence_count"] = np.where(
        df["text_len_chars"] > 0,
        sentence_count + 1,
        0,
    ).astype("int16")

    df["avg_sentence_len_words"] = (
        (df["text_len_words"] / df["sentence_count"].replace(0, np.nan))
        .fillna(0.0)
        .astype("float32")
    )

    df["ellipsis_count"] = txt.str.count(r"\.\.\.|…").astype("int16")
    df["repeat_punct_count"] = txt.str.count(r"[!?]{2,}").astype("int16")

    df["has_url"] = txt.str.contains(r"http|www|vk\.cc", regex=True, na=False).astype(
        "int8"
    )
    df["has_number"] = txt.str.contains(r"\d", regex=True, na=False).astype("int8")
    df["has_price"] = txt.str.contains(
        r"\d+\s?(₽|руб|р\b)|скидк|цена|бесплатно|акция",
        regex=True,
        case=False,
        na=False,
    ).astype("int8")

    df["starts_with_question"] = txt.str.match(
        r"^\s*[^a-zA-ZА-Яа-яЁё0-9]*.*\?", na=False
    ).astype("int8")
    df["starts_with_number"] = txt.str.match(r"^\s*\d", na=False).astype("int8")
    df["starts_with_emoji"] = txt.str.match(
        r"^\s*[\U0001F300-\U0001FAFF]", na=False
    ).astype("int8")

    cta_pattern = (
        r"пиши|смотри|читай|успей|переходи|жми|сохрани|подпишись|голосуй|оцени"
    )
    urgency_pattern = r"сегодня|сейчас|срочно|только сегодня|последний шанс|успей"

    df["has_call_to_action"] = txt.str.contains(
        cta_pattern,
        regex=True,
        case=False,
        na=False,
    ).astype("int8")

    df["has_urgency"] = txt.str.contains(
        urgency_pattern,
        regex=True,
        case=False,
        na=False,
    ).astype("int8")

    df["part_of_day"] = pd.cut(
        df["hour"],
        bins=[-1, 5, 11, 17, 23],
        labels=["night", "morning", "afternoon", "evening"],
    )

    df["text_len_bin"] = pd.cut(
        df["text_len_words"],
        bins=[-1, 5, 15, 30, 60, 10**9],
        labels=["very_short", "short", "medium", "long", "very_long"],
    )

    df["hour_weekday"] = df["hour"].astype(str) + "_" + df["weekday"].astype(str)
    df["domain_weekday"] = (
        df["domain"].fillna("unknown").astype(str) + "_" + df["weekday"].astype(str)
    )
    df["domain_part_of_day"] = (
        df["domain"].fillna("unknown").astype(str) + "_" + df["part_of_day"].astype(str)
    )
    df["weekend_hour"] = df["is_weekend"].astype(str) + "_" + df["hour"].astype(str)

    return df

##Подготовка датасета

In [ ]:
data = make_features(df)

In [ ]:
data.head()

### Метрики

In [ ]:
def mape_score(y_true, y_pred, eps=1e-8):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100

def print_metrics(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mape_score(y_true, y_pred)

    print(f"{name}:")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")
    print(f"  MAPE = {mape:.2f}%")
    print()




In [ ]:
def print_metrics_er(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"{name}:")
    print(f"  RMSE = {rmse:.4f}")
    print(f"  MAE  = {mae:.4f}")
    print(f"  R2   = {r2:.4f}")
    print()

##Бейзлайн - таргет просмотры

In [ ]:
num_cols = ['n_photos',
       'hour', 'weekday', 'month', 'day', 'is_weekend',
       'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'text_len_chars',
       'text_len_words', 'num_lines', 'num_exclam', 'num_question', 'num_dots',
       'num_commas', 'num_colons', 'num_hashtags', 'num_links', 'has_mention',
       'num_emojis', 'caps_ratio', 'unique_words_count',
       'unique_words_ratio', 'avg_word_len', 'long_words_count', 'digit_count',
       'digit_ratio', 'uppercase_words_count', 'sentence_count',
       'avg_sentence_len_words', 'ellipsis_count', 'repeat_punct_count',
       'has_url', 'has_number', 'has_price', 'starts_with_question',
       'starts_with_number', 'starts_with_emoji', 'has_call_to_action',
       'has_urgency', 'is_holiday','is_workday','is_friday']

text_col = "text"
target_col = "target_views"
cat_cols = [
    "domain",
    "part_of_day",
    "text_len_bin",
    "hour_weekday",
    "domain_weekday",
    "domain_part_of_day",
    "weekend_hour"
]

In [ ]:
X = data[[text_col] + num_cols + cat_cols].copy()
y = data[target_col].values

# 70% train, 15% valid, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=50000,
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.95,
                sublinear_tf=True
            ),
            text_col
        ),
        (
            "num",
            StandardScaler(),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols
        )
    ],
    remainder="drop"
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_valid = model.predict(X_valid)
y_pred_test = model.predict(X_test)


y_train_real = np.expm1(y_train)
y_valid_real = np.expm1(y_valid)
y_test_real = np.expm1(y_test)

y_pred_train_real = np.expm1(y_pred_train)
y_pred_valid_real = np.expm1(y_pred_valid)
y_pred_test_real = np.expm1(y_pred_test)

print_metrics(y_train_real, y_pred_train_real, "Train")
print_metrics(y_valid_real, y_pred_valid_real, "Validation")
print_metrics(y_test_real, y_pred_test_real, "Test")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
X_train_rf = X_train.drop(columns=[text_col]).copy()
X_valid_rf = X_valid.drop(columns=[text_col]).copy()
X_test_rf = X_test.drop(columns=[text_col]).copy()

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

rf_model = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])


rf_model.fit(X_train_rf, y_train)

y_pred_train_rf = rf_model.predict(X_train_rf)
y_pred_valid_rf = rf_model.predict(X_valid_rf)
y_pred_test_rf = rf_model.predict(X_test_rf)


y_pred_train_rf_real = np.expm1(y_pred_train_rf)
y_pred_valid_rf_real = np.expm1(y_pred_valid_rf)
y_pred_test_rf_real = np.expm1(y_pred_test_rf)


print_metrics(y_train_real, y_pred_train_rf_real, "RF Train")
print_metrics(y_valid_real, y_pred_valid_rf_real, "RF Validation")
print_metrics(y_test_real, y_pred_test_rf_real, "RF Test")

##Катбуст -таргет просмотры

In [ ]:

text_cols = ["text"]
cat_cols = [
    "domain", "part_of_day", "text_len_bin", "hour_weekday",
    "domain_weekday", "domain_part_of_day", "weekend_hour"
]
num_cols = [
     "n_photos", "hour", "weekday", "month", "day", "is_weekend",
    "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "text_len_chars",
    "text_len_words", "num_lines", "num_exclam", "num_question", "num_dots",
    "num_commas", "num_colons", "num_hashtags", "num_links", "has_mention",
    "num_emojis", "caps_ratio", "unique_words_count",
    "unique_words_ratio", "avg_word_len", "long_words_count", "digit_count",
    "digit_ratio", "uppercase_words_count", "sentence_count",
    "avg_sentence_len_words", "ellipsis_count", "repeat_punct_count",
    "has_url", "has_number", "has_price", "starts_with_question",
    "starts_with_number", "starts_with_emoji", "has_call_to_action",
    "has_urgency", "is_holiday", "is_workday", "is_friday"
]
feature_cols = text_cols + num_cols + cat_cols

In [ ]:

X_train_cb = X_train[feature_cols].copy()
X_valid_cb = X_valid[feature_cols].copy()
X_test_cb = X_test[feature_cols].copy()

train_pool = Pool(
    data=X_train_cb,
    label=y_train,
    text_features=text_cols,
    cat_features=cat_cols
)

valid_pool = Pool(
    data=X_valid_cb,
    label=y_valid,
    text_features=text_cols,
    cat_features=cat_cols
)

test_pool = Pool(
    data=X_test_cb,
    label=y_test,
    text_features=text_cols,
    cat_features=cat_cols
)

model_cb = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    early_stopping_rounds=200,
    verbose=300
)

model_cb.fit(
    train_pool,
    eval_set=valid_pool,
    use_best_model=True
)

y_pred_train_cb = model_cb.predict(train_pool)
y_pred_valid_cb = model_cb.predict(valid_pool)
y_pred_test_cb = model_cb.predict(test_pool)

y_pred_train_cb_real = np.expm1(y_pred_train_cb)
y_pred_valid_cb_real = np.expm1(y_pred_valid_cb)
y_pred_test_cb_real = np.expm1(y_pred_test_cb)



print_metrics(y_train_real, y_pred_train_cb_real, "CB Train")
print_metrics(y_valid_real, y_pred_valid_cb_real, "CB Validation")
print_metrics(y_test_real, y_pred_test_cb_real, "CB Test")

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

MODEL_NAME = "cointegrated/rubert-tiny2"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

BATCH_SIZE = 64
MAX_LENGTH = 512

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_text = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model_text.eval()

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * input_mask_expanded).sum(1) / input_mask_expanded.sum(1).clamp(min=1e-9)

In [ ]:
@torch.no_grad()
def encode_texts(texts, tokenizer, model, batch_size=64, max_length=128, device="cpu"):
    all_embeddings = []

    texts = pd.Series(texts).fillna("").astype(str).tolist()

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = model(**encoded)

        emb = mean_pooling(outputs, encoded["attention_mask"])
        emb = F.normalize(emb, p=2, dim=1)
        all_embeddings.append(emb.cpu().numpy())

    return np.vstack(all_embeddings)

In [ ]:
text_cols = ["text"]

cat_cols = [
    "domain", "part_of_day", "text_len_bin", "hour_weekday",
    "domain_weekday", "domain_part_of_day", "weekend_hour"
]
num_cols = [
    "n_photos", "hour", "weekday", "month", "day", "is_weekend",
    "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "text_len_chars",
    "text_len_words", "num_lines", "num_exclam", "num_question", "num_dots",
    "num_commas", "num_colons", "num_hashtags", "num_links", "has_mention",
    "num_emojis", "caps_ratio","unique_words_count",
    "unique_words_ratio", "avg_word_len", "long_words_count", "digit_count",
    "digit_ratio", "uppercase_words_count", "sentence_count",
    "avg_sentence_len_words", "ellipsis_count", "repeat_punct_count",
    "has_url", "has_number", "has_price", "starts_with_question",
    "starts_with_number", "starts_with_emoji", "has_call_to_action",
    "has_urgency", "is_holiday", "is_workday", "is_friday"
]

feature_cols = text_cols + num_cols + cat_cols




X_train_cb = X_train[feature_cols].copy()
X_valid_cb = X_valid[feature_cols].copy()
X_test_cb = X_test[feature_cols].copy()

train_emb = encode_texts(
    X_train_cb["text"],
    tokenizer=tokenizer,
    model=model_text,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE
)

valid_emb = encode_texts(
    X_valid_cb["text"],
    tokenizer=tokenizer,
    model=model_text,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE
)

test_emb = encode_texts(
    X_test_cb["text"],
    tokenizer=tokenizer,
    model=model_text,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE
)

print(train_emb.shape, valid_emb.shape, test_emb.shape)

In [ ]:
emb_cols = [f"rubert_emb_{i}" for i in range(train_emb.shape[1])]

train_emb_df = pd.DataFrame(train_emb, index=X_train_cb.index, columns=emb_cols)
valid_emb_df = pd.DataFrame(valid_emb, index=X_valid_cb.index, columns=emb_cols)
test_emb_df  = pd.DataFrame(test_emb,  index=X_test_cb.index,  columns=emb_cols)

In [ ]:
X_train_cb_emb = pd.concat(
    [X_train_cb.drop(columns=["text"]).reset_index(drop=True),
     train_emb_df.reset_index(drop=True)],
    axis=1
)

X_valid_cb_emb = pd.concat(
    [X_valid_cb.drop(columns=["text"]).reset_index(drop=True),
     valid_emb_df.reset_index(drop=True)],
    axis=1
)

X_test_cb_emb = pd.concat(
    [X_test_cb.drop(columns=["text"]).reset_index(drop=True),
     test_emb_df.reset_index(drop=True)],
    axis=1
)

In [ ]:
train_pool_emb = Pool(
    data=X_train_cb_emb,
    label=y_train,
    cat_features=cat_cols
)

valid_pool_emb = Pool(
    data=X_valid_cb_emb,
    label=y_valid,
    cat_features=cat_cols
)

test_pool_emb = Pool(
    data=X_test_cb_emb,
    label=y_test,
    cat_features=cat_cols
)

In [ ]:
model_cb_emb = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    early_stopping_rounds=200,
    verbose=100
)

model_cb_emb.fit(
    train_pool_emb,
    eval_set=valid_pool_emb,
    use_best_model=True
)

In [ ]:
y_pred_train_cb_emb = model_cb_emb.predict(train_pool_emb)
y_pred_valid_cb_emb = model_cb_emb.predict(valid_pool_emb)
y_pred_test_cb_emb = model_cb_emb.predict(test_pool_emb)

y_pred_train_cb_emb_real = np.maximum(np.expm1(y_pred_train_cb_emb), 0)
y_pred_valid_cb_emb_real = np.maximum(np.expm1(y_pred_valid_cb_emb), 0)
y_pred_test_cb_emb_real = np.maximum(np.expm1(y_pred_test_cb_emb), 0)

print_metrics(y_train_real, y_pred_train_cb_emb_real, "CB+RuBERT Train")
print_metrics(y_valid_real, y_pred_valid_cb_emb_real, "CB+RuBERT Validation")
print_metrics(y_test_real, y_pred_test_cb_emb_real, "CB+RuBERT Test")

In [ ]:
print(model_cb_emb.get_feature_importance(prettified=True).head(50))

In [ ]:
X_train_tabnet = pd.concat(
    [
        X_train_cb[num_cols + cat_cols].reset_index(drop=True),
        train_emb_df.reset_index(drop=True)
    ],
    axis=1
)

X_valid_tabnet = pd.concat(
    [
        X_valid_cb[num_cols + cat_cols].reset_index(drop=True),
        valid_emb_df.reset_index(drop=True)
    ],
    axis=1
)

X_test_tabnet = pd.concat(
    [
        X_test_cb[num_cols + cat_cols].reset_index(drop=True),
        test_emb_df.reset_index(drop=True)
    ],
    axis=1
)

In [ ]:
pip install pytorch_tabnet

In [ ]:
from sklearn.preprocessing import LabelEncoder
from pytorch_tabnet.tab_model import TabNetRegressor

In [ ]:
X_train_tabnet = X_train_tabnet.copy()
X_valid_tabnet = X_valid_tabnet.copy()
X_test_tabnet = X_test_tabnet.copy()

cat_idxs = []
cat_dims = []

for col in cat_cols:
    le = LabelEncoder()

    train_vals = X_train_tabnet[col].astype(str).fillna("NA")
    valid_vals = X_valid_tabnet[col].astype(str).fillna("NA")
    test_vals = X_test_tabnet[col].astype(str).fillna("NA")

    all_vals = pd.concat([train_vals, valid_vals, test_vals], axis=0)
    le.fit(all_vals)

    X_train_tabnet[col] = le.transform(train_vals)
    X_valid_tabnet[col] = le.transform(valid_vals)
    X_test_tabnet[col] = le.transform(test_vals)

    cat_idxs.append(X_train_tabnet.columns.get_loc(col))
    cat_dims.append(len(le.classes_))

In [ ]:
X_train_tabnet_np = X_train_tabnet.astype(np.float32).values
X_valid_tabnet_np = X_valid_tabnet.astype(np.float32).values
X_test_tabnet_np = X_test_tabnet.astype(np.float32).values

y_train_tabnet = y_train.reshape(-1, 1).astype(np.float32)
y_valid_tabnet = y_valid.reshape(-1, 1).astype(np.float32)
y_test_tabnet = y_test.reshape(-1, 1).astype(np.float32)

print(X_train_tabnet_np.shape, X_valid_tabnet_np.shape, X_test_tabnet_np.shape)
print(cat_idxs)
print(cat_dims)

In [ ]:
tabnet_model = TabNetRegressor(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=2,
    n_d=16,
    n_a=16,
    n_steps=3,
    gamma=1.3,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=1e-3),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={"step_size": 30, "gamma": 0.95},
    mask_type="entmax",
    seed=42,
    verbose=10
)

In [ ]:
tabnet_model.fit(
    X_train=X_train_tabnet_np,
    y_train=y_train_tabnet,
    eval_set=[(X_valid_tabnet_np, y_valid_tabnet)],
    eval_name=["valid"],
    eval_metric=["rmse"],
    max_epochs=200,
    patience=20,
    batch_size=128,
    virtual_batch_size=64,
    num_workers=0,
    drop_last=False
)

In [ ]:
y_pred_train_tabnet = tabnet_model.predict(X_train_tabnet_np).ravel()
y_pred_valid_tabnet = tabnet_model.predict(X_valid_tabnet_np).ravel()
y_pred_test_tabnet = tabnet_model.predict(X_test_tabnet_np).ravel()

In [ ]:
y_pred_train_tabnet_real = np.expm1(y_pred_train_tabnet)
y_pred_valid_tabnet_real = np.expm1(y_pred_valid_tabnet)
y_pred_test_tabnet_real = np.expm1(y_pred_test_tabnet)

print_metrics(y_train_real, y_pred_train_tabnet_real, "TabNet Train")
print_metrics(y_valid_real, y_pred_valid_tabnet_real, "TabNet Validation")
print_metrics(y_test_real, y_pred_test_tabnet_real, "TabNet Test")

In [ ]:
base_cols = num_cols + cat_cols

X_train_cb_emb = X_train[base_cols].copy()
X_valid_cb_emb = X_valid[base_cols].copy()
X_test_cb_emb = X_test[base_cols].copy()

X_train_cb_emb["rubert_embedding"] = list(train_emb)
X_valid_cb_emb["rubert_embedding"] = list(valid_emb)
X_test_cb_emb["rubert_embedding"] = list(test_emb)

In [ ]:
train_pool_emb = Pool(
    data=X_train_cb_emb,
    label=y_train,
    cat_features=cat_cols,
    embedding_features=["rubert_embedding"]
)

valid_pool_emb = Pool(
    data=X_valid_cb_emb,
    label=y_valid,
    cat_features=cat_cols,
    embedding_features=["rubert_embedding"]
)

test_pool_emb = Pool(
    data=X_test_cb_emb,
    label=y_test,
    cat_features=cat_cols,
    embedding_features=["rubert_embedding"]
)

In [ ]:
model_cb_emb = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    early_stopping_rounds=200,
    verbose=100
)

model_cb_emb.fit(
    train_pool_emb,
    eval_set=valid_pool_emb,
    use_best_model=True
)

In [ ]:
y_pred_train_cb_emb = model_cb_emb.predict(train_pool_emb)
y_pred_valid_cb_emb = model_cb_emb.predict(valid_pool_emb)
y_pred_test_cb_emb = model_cb_emb.predict(test_pool_emb)

y_pred_train_cb_emb_real = np.maximum(np.expm1(y_pred_train_cb_emb), 0)
y_pred_valid_cb_emb_real = np.maximum(np.expm1(y_pred_valid_cb_emb), 0)
y_pred_test_cb_emb_real = np.maximum(np.expm1(y_pred_test_cb_emb), 0)

print_metrics(y_train_real, y_pred_train_cb_emb_real, "CB+Emb Train")
print_metrics(y_valid_real, y_pred_valid_cb_emb_real, "CB+Emb Validation")
print_metrics(y_test_real, y_pred_test_cb_emb_real, "CB+Emb Test")

In [ ]:
print(model_cb_emb.get_feature_importance(prettified=True).head(30))

In [ ]:
#Обучение lightautoml
automl = TabularNLPAutoML(
    task=Task("reg"),
    timeout=3600,
    cpu_limit=4,
    memory_limit=16,
    gpu_ids="0",

    text_params={
        "lang": "ru",
        "bert_model": "cointegrated/rubert-tiny",
        "max_length": 512,
        "batch_size": 32,
    },

    tfidf_params={
        "max_features": 30000,
        "ngram_range": (1, 2),
        "min_df": 3,
    },

    autonlp_params={
        "sent_scaler": "l2",
        "pooling": "mean",
        "use_tfidf": True,
        "use_bert": True,
        "use_fasttext": False,
    }
)

roles = {
    "target": "target_views",
    "text": ["text"],
    "drop": ["engagement_rate", "dt_msk",'post_id']
}

# 70 / 15 / 15
train_data, temp_data = train_test_split(
    data,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

valid_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

valid_pred = automl.fit_predict(
    train_data,
    roles=roles,
    valid_data=valid_data,
    verbose=10
)


train_pred = automl.predict(train_data)
test_pred = automl.predict(test_data)

In [ ]:
y_train = train_data["target_views"].values
y_valid = valid_data["target_views"].values
y_test = test_data["target_views"].values

y_pred_train = train_pred.data[:, 0]
y_pred_valid = valid_pred.data[:, 0]
y_pred_test = test_pred.data[:, 0]

y_train_real = np.expm1(y_train)
y_valid_real = np.expm1(y_valid)
y_test_real = np.expm1(y_test)

y_pred_train_real = np.maximum(np.expm1(y_pred_train), 0)
y_pred_valid_real = np.maximum(np.expm1(y_pred_valid), 0)
y_pred_test_real = np.maximum(np.expm1(y_pred_test), 0)

print_metrics(y_train_real, y_pred_train_real, "Train")
print_metrics(y_valid_real, y_pred_valid_real, "Validation")
print_metrics(y_test_real, y_pred_test_real, "Test")

## AUTOML на просмотры


In [ ]:
##automl

In [ ]:
automl = TabularNLPAutoML(
    task=Task("reg"),
    timeout=3600,
    cpu_limit=4,
    memory_limit=16,
    gpu_ids="0",

    text_params={
        "lang": "ru",
        "bert_model": "cointegrated/rubert-tiny",
        "max_length": 512,
        "batch_size": 32,
    },

    tfidf_params={
        "max_features": 30000,
        "ngram_range": (1, 2),
        "min_df": 3,
    },

    autonlp_params={
        "sent_scaler": "l2",
        "pooling": "mean",
        "use_tfidf": True,
        "use_bert": True,
        "use_fasttext": False,
    }
)

roles = {
    "target": "target_views",
    "text": ["text"],
    "drop": ["engagement_rate", "dt_msk",'post_id']
}

train_data, temp_data = train_test_split(
    data,
    test_size=0.30,
    random_state=43,
    shuffle=True
)

valid_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=43,
    shuffle=True
)

valid_pred = automl.fit_predict(
    train_data,
    roles=roles,
    valid_data=valid_data,
    verbose=10
)


train_pred = automl.predict(train_data)
test_pred = automl.predict(test_data)

In [ ]:
y_train = train_data["target_views"].values
y_valid = valid_data["target_views"].values
y_test = test_data["target_views"].values

y_pred_train = train_pred.data[:, 0]
y_pred_valid = valid_pred.data[:, 0]
y_pred_test = test_pred.data[:, 0]

y_train_real = np.expm1(y_train)
y_valid_real = np.expm1(y_valid)
y_test_real = np.expm1(y_test)

y_pred_train_real = np.maximum(np.expm1(y_pred_train), 0)
y_pred_valid_real = np.maximum(np.expm1(y_pred_valid), 0)
y_pred_test_real = np.maximum(np.expm1(y_pred_test), 0)

print_metrics(y_train_real, y_pred_train_real, "Train")
print_metrics(y_valid_real, y_pred_valid_real, "Validation")
print_metrics(y_test_real, y_pred_test_real, "Test")